# Plot figure showing users and credentialed users

## Set up

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tableone import TableOne
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# path to datasets
base_path = os.path.join("..", "data", "physionet")

## Load the data

In [ ]:
# Custom function to parse the datetime
def parse_publish_date(date_str):
    try:
        # Try the format with microseconds first
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S.%f%z")
    except ValueError:
        # If that fails, try the format without microseconds
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S%z")

In [ ]:
users = pd.read_csv(os.path.join(base_path, "users.csv"), low_memory=False)

# Convert date columns to date type
users["join_date"] = pd.to_datetime(
    users["join_date"],
    format="%Y-%m-%d",
    errors="raise"
)

In [ ]:
# Filter out 2026
users = users[users["join_date"] < "2026-01-01"]

In [ ]:
# Users over time
users['join_date'].dt.year.unique()

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Extract the year from 'join_date' and create a new column 'year'
users['year'] = users['join_date'].dt.year

# Group by 'year' and count the number of users per year
users_per_year = users.groupby('year').agg(count=('user_id', 'size')).reset_index()

users_per_year

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Filter the DataFrame for credentialed users using .loc[]
credentialed_users = users.loc[users['credentialing_status'] == "Credentialed"].copy()

# Extract the year from 'join_date' and create a new column 'year' using .loc[]
credentialed_users.loc[:, 'year'] = credentialed_users['join_date'].dt.year

# Group by 'year' and count the number of users per year
credentialed_users_per_year = credentialed_users.groupby('year').agg(count=('user_id', 'size')).reset_index()

credentialed_users_per_year

## Plot functions

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_registered_credentialed_users_modern(
        users_per_year,
        credentialed_users_per_year,
        size_title=32,
        size_axes_labels=30,
        size_tick_labels=22,
        incomplete=True,
        bold=False):

    accent = "#4c72b0"
    accent_faded = "rgba(76,114,176,0.35)"
    text_col = "#222"
    tick_col = "#555"

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("New Registered Users", "New Credentialed Users"),
        horizontal_spacing=0.12,
    )

    # ----------------------------------------------------------
    # Helper: add star above final bar
    # ----------------------------------------------------------
    def add_star(fig, years, counts, row, col, star_size=18):
        final_year = years.iloc[-1]
        final_val = counts.iloc[-1]
        fig.add_trace(
            go.Scatter(
                x=[final_year],
                y=[final_val * 1.05],
                mode="text",
                text=["★"],
                textfont=dict(size=star_size, color="#c0392b"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row, col=col
        )

    # ----------------------------------------------------------
    # Panel 1: Registered Users
    # ----------------------------------------------------------
    colors_reg = []
    for i in range(len(users_per_year)):
        if incomplete and i == len(users_per_year) - 1:
            colors_reg.append(accent_faded)
        else:
            colors_reg.append(accent)

    fig.add_trace(
        go.Bar(
            x=users_per_year["year"],
            y=users_per_year["count"],
            marker=dict(color=colors_reg,
                        line=dict(color=accent, width=0.5)),
            width=0.55,
            name="Registered Users",
        ),
        row=1, col=1
    )

    if incomplete:
        add_star(fig, users_per_year["year"], users_per_year["count"], row=1, col=1)

    # ----------------------------------------------------------
    # Panel 2: Credentialed Users
    # ----------------------------------------------------------
    colors_cred = []
    for i in range(len(credentialed_users_per_year)):
        if incomplete and i == len(credentialed_users_per_year) - 1:
            colors_cred.append(accent_faded)
        else:
            colors_cred.append(accent)

    fig.add_trace(
        go.Bar(
            x=credentialed_users_per_year["year"],
            y=credentialed_users_per_year["count"],
            marker=dict(color=colors_cred,
                        line=dict(color=accent, width=0.5)),
            width=0.55,
            name="Credentialed Users",
        ),
        row=1, col=2
    )

    if incomplete:
        add_star(fig, credentialed_users_per_year["year"],
                 credentialed_users_per_year["count"], row=1, col=2)

    # ----------------------------------------------------------
    # Layout + fonts
    # ----------------------------------------------------------
    fig.update_layout(
        template="simple_white",
        font=dict(family="Arial", size=size_tick_labels, color=text_col),
        showlegend=False,
        margin=dict(l=80, r=40, t=120, b=80),  # t increased from 90 → 120
        height=650,
        width=1400,
    )

    # ----------------------------------------------------------
    # Replace subplot titles with properly centred, non-bold ones
    # ----------------------------------------------------------
    fig.update_layout(
        annotations=[
            dict(
                text="New Registered Users Per Year",
                x=0.225, y=1.10,                # centered above left subplot
                xanchor="center",
                font=dict(size=size_title, family="Arial", color=text_col),
                showarrow=False
            ),
            dict(
                text="New Credentialed Users Per Year",
                x=0.775, y=1.10,                # centered above right subplot
                xanchor="center",
                font=dict(size=size_title, family="Arial", color=text_col),
                showarrow=False
            ),
        ]
    )

    # ----------------------------------------------------------
    # Axes styling
    # ----------------------------------------------------------
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_labels, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="New Registered Users",
        title_font=dict(size=size_axes_labels, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        separatethousands=True,
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        row=1, col=1
    )

    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_labels, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="New Credentialed Users",
        title_font=dict(size=size_axes_labels, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        separatethousands=True,
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        row=1, col=2
    )

    return fig


## Plot them!

In [ ]:
fig = plot_registered_credentialed_users_modern(users_per_year,
                                          credentialed_users_per_year,
                                          incomplete=False)

fig.write_image("../figures/figure_4_users.png")
fig.write_image("../figures/figure_4_users.svg")

In [ ]:
print(users_per_year.columns)
print(users_per_year.head())